# LangChain Retrieval Augmentation



<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/langchain-retrieval-augmentation-202503.ipynb
</div>

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>

## Environment Setup

<br>

> 💡 **Important:**
>
> This notebook requires **LangChain < 0.3**.
>
> Below, you will find two options for installing the required dependencies. Choose the one that best matches your environment::
>
> 🖥️ **Running locally?**  
> → Use **Option A** to create a dedicated virtual environment (recommended).
>
> ☁️ **Using Google Colab or want a quick setup?**  
> → Use **Option B** to install the required dependencies directly.
>

<br><br>


### Option A: Use a virtual environment

Open a terminal and run the following commands.


<br>

**1. Create a virtual environment:**

```bash
python -m venv .venv/langchain-v0.2.x
```

<br>

**2. Activate the virtual environment:**

- **macOS / Linux:**
```bash
    source .venv/langchain-v0.2.x/bin/activate
```

- **Windows (PowerShell):**
```powershell
    .\.venv\langchain-v0.2.x\Scripts\Activate.ps1
```

<br>

**3. Install the required packages:**

```bash
python -m pip install \
    "langchain<0.3" \
    "langchain-core<0.3" \
    "langchain-community<0.3" \
    "langchain-openai<0.2" \
    ipykernel
```

<br>

**4. Register the environment as a Jupyter kernel:**

```bash
python -m ipykernel install \
    --user \
    --name langchain-v0.2.x \
    --display-name "Python (LangChain 0.2.x)"
```

<br>

**5. Select the correct kernel:**

Once you've completed the previous steps, do the following:
1. Open this notebook in your favourite environment (e.g., Jupyter or VS Code)
2. Select the Kernel you've just created
    - **Jupyter**: Kernel → Change Kernel → Python (LangChain 0.2.x).
    - **VS Code**: Click the Kernel selector in the top-right corner of the notebook editor → Jupyter Kernel → Python (LangChain 0.2.x)
        - Notice that you need to select "Jupyter Kernel" (not "Python Environments")
        - If "Python (LangChain 0.2.x)" doesn't appear in the kernel list, reload VS Code:
            - Press Cmd + Shift + P to open the Command Palette
            - Type Developer: Reload Window and press Enter
            - Try selecting the kernel again
3. Run the notebook as usual.

<br>

> **Notes:**
>
> - Make sure to add the directory `.venv` to your `.gitignore`
> - The virtual environment setup only needs to be completed once. The environment can then be reused for other notebooks that require **LangChain < 0.3**, without affecting your default Python environment or newer LangChain installations.

<br>

### Option B: Install dependencies directly

If you are using Google Colab, or you're having problems configuring a virtual environment, create a code cell and run the command below:

```python
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"
```

<br>


</div>

<br>

## Intro to RAG

<br>

### The problem: LLMs have limited, frozen knowledge

Large Language Models (LLMs) are trained on a large snapshot of text collected up to a certain point in time. Once training is finished, that knowledge is **frozen**: the model has no built-in way to learn about things that happened afterwards, and no visibility into private or domain-specific data (like your company's internal documents) that were never part of its training set.

This leads to two well-known limitations:

- **Data freshness**: the model can't know about recent events, released after its training cutoff.
- **Missing/private knowledge**: the model has no access to information it was never trained on (internal docs, proprietary databases, personal notes, etc.).

When an LLM doesn't know an answer, it may either say so, or — more problematically — it may confidently generate a plausible-sounding but incorrect answer. This is known as **hallucination**.

> 💡 **Note:** Many frontier models now have access to tools like **web search**, which lets them fetch live information from the internet to work around the data-freshness problem. This is useful for public, general-purpose information, but it doesn't solve everything:
> - It doesn't give the model access to **private or proprietary data** (internal company documents, personal files, restricted databases) that isn't published on the web.
> - It offers **less control** over *which* sources are used, how they're ranked, and how they're formatted for the model.
> - It can be **slower and less reliable** than querying a purpose-built knowledge base you control.
>
> RAG is a complementary (and often preferable) technique for these cases: instead of searching the open web, we build and query *our own* knowledge base, giving us full control over the content, quality, and traceability of the sources used to ground the model's answers.

<br>

### The solution: Retrieval-Augmented Generation (RAG)

**Retrieval-Augmented Generation (RAG)** addresses this by connecting the LLM to an external knowledge source at query time, rather than relying solely on what it memorized during training, or on a general-purpose web search.

The core idea is simple:

1. **Store** your knowledge (documents, articles, internal data, etc.) in a searchable knowledge base — typically as *embeddings* in a vector database.
2. **Retrieve** the pieces of information most relevant to the user's question from that knowledge base.
3. **Augment** the prompt sent to the LLM by including that retrieved information alongside the original question.
4. **Generate** a response — the LLM now answers using both its general language abilities *and* the fresh, specific context it was just given.

<br>

![](../_images/rag_1.webp)

<br>

![](../_images/rag_2.jpeg)

<br>

This approach gives us several benefits:

- ✅ **Up-to-date answers**, without having to retrain the model.
- ✅ **Access to private/custom data**, without exposing it during training.
- ✅ **Reduced hallucinations**, since the model is grounded in real retrieved text.
- ✅ **Traceability**, since we can cite exactly which source(s) the answer came from.




<br>

## How a RAG Pipeline Works

A RAG pipeline is made up of two main phases: an **offline data preparation (indexing)** phase, done once (or whenever your data changes), and an **online query-time (retrieval + generation)** phase, done every time we process a request (e.g., when a user asks a question).
<br>

![](../_images/rag_pipeline.png)

<br>

### Phase 1: Data Preparation (Indexing)

Before we can retrieve anything, we need to build the knowledge base:

1. **Load documents**: gather the raw source material (articles, PDFs, web pages, internal docs, etc.).
2. **Split into chunks**: break each document into smaller, manageable pieces of text. This matters because embedding models and LLMs work best with focused, reasonably-sized pieces of text, and it lets us retrieve just the specific passages relevant to a question instead of an entire document.
3. **Generate embeddings**: convert each chunk of text into a numerical vector (an *embedding*) that captures its meaning, using an embedding model.
4. **Store in a vector database**: save each chunk's embedding, along with its original text and metadata, in a vector database, so it can be searched efficiently later.

<br>

### Phase 2: Retrieval & Generation (Query Time)

This happens every time a user asks a question:

1. **Embed the query**: convert the user's question into an embedding, using the *same* embedding model used during indexing.
2. **Retrieve relevant chunks**: search the vector database for the chunks whose embeddings are most similar to the query embedding (i.e. the most semantically relevant pieces of text).
3. **Augment the prompt**: insert the retrieved chunks into the prompt sent to the LLM, together with the original question, so the model has the specific context it needs.
4. **Generate the answer**: the LLM produces a response grounded in the retrieved context, rather than relying only on what it memorized during training.

<br>

In short: **indexing** builds a searchable knowledge base ahead of time, and **retrieval + generation** uses that knowledge base at query time to give the LLM the right context to answer accurately.


<br>

## Intro to this Demo

Now that we understand the theory, let's put it into practice. In this notebook, we'll build a complete RAG pipeline end-to-end using LangChain, following the exact two phases described above:


- **Indexing**: we'll load a Wikipedia dataset, split the articles into chunks, generate embeddings for each chunk, and store them in a vector database (Pinecone).

- **Retrieval & generation**: we'll embed a user's question, retrieve the most relevant chunks from the vector database, and use them to generate a grounded answer with an LLM — including citing the sources used.By the end, you'll have a working question-answering system that can accurately answer questions using knowledge it was never explicitly trained on.


<br>

## Load Environment Variables


In [1]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')


<br>

### Load the data

For this demo, we'll use the [`wikimedia/wikipedia`](https://huggingface.co/datasets/wikimedia/wikipedia) dataset from Hugging Face, which contains articles from **Simple English Wikipedia** (a version of Wikipedia written using simpler vocabulary and grammar).

We'll only load the first **10,000 articles** (`train[:10000]`) to keep things fast for this demo. Each record includes:

- `id`: a unique identifier for the article
- `url`: a link to the article on Wikipedia
- `title`: the article's title
- `text`: the full plain-text content of the article

This gives us a realistic, ready-made collection of documents to act as our knowledge base — similar to what you'd get from any other source (PDFs, company policies, internal docs, web pages, etc.) in a real-world RAG project.

In [2]:
%pip install datasets


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from datasets import load_dataset

data = load_dataset("wikimedia/wikipedia", "20231101.simple", split='train[:10000]')
data

/Users/luis/Desktop/ironhack_june26/1_ai_eng_lectures/.venv/langchain-v0.2.x/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset({
    features: ['id', 'url', 'title', 'text'],
    num_rows: 10000
})

In [ ]:
# let's pick one article as an example
example_article = data[6]

example_article

<br>

Now we install the remaining libraries:

In [5]:
%pip install -qU "langchain-pinecone<0.2" "pinecone-notebooks==0.1.1"


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install --upgrade --quiet "langchain-text-splitters<0.3" "tiktoken==0.13.0"


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


<br>

### Split documents into chunks

Before we can embed and store our articles, we need to split each one into smaller chunks of text.

We do this for two main reasons:

- **Embedding models and LLMs work best with focused, reasonably-sized pieces of text**, rather than an entire article at once.
- **It lets us retrieve just the specific passage relevant to a question**, instead of pulling back an entire document every time.

Since embedding models (and LLMs in general) measure input size in **tokens** rather than characters or words, we first need a way to count how many tokens a piece of text will use. To do this, we'll use `tiktoken` — OpenAI's tokenizer — to define a length function that we can plug into our text splitter.

In [7]:
import tiktoken

tiktoken.encoding_for_model('gpt-3.5-turbo')

<Encoding 'cl100k_base'>

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding('cl100k_base')

# create the length function
def tiktoken_len(text):
    tokens = tokenizer.encode(
        text,
        disallowed_special=()
    )
    return len(tokens)

amount_1 = tiktoken_len("the quick brown fox")
amount_2 = tiktoken_len("hello I am a chunk of text and using the tiktoken_len function "
             "we can find the length of this chunk of text in tokens")

print(amount_1)
print(amount_2)

26

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# create a text splitter that breaks documents into token-sized chunks,
# trying to split on natural boundaries (paragraphs, then lines, then words) first
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # max chunk size, measured in tokens (via tiktoken_len)
    chunk_overlap=20,  # tokens shared between consecutive chunks, to preserve context across boundaries
    length_function=tiktoken_len,  # use our tiktoken-based function to measure chunk size in tokens
    separators=["\n\n", "\n", " ", ""]  # try splitting on these, in order, until chunks fit chunk_size
)

In [10]:
# as an example, let's split one article into chunks using our text_splitter
chunks = text_splitter.split_text(example_article['text'])

chunks

['Alan Mathison Turing OBE FRS (London, 23 June 1912 – Wilmslow, Cheshire, 7 June 1954) was an English mathematician and computer scientist. He was born in Maida Vale, London.\n\nEarly life and family \nAlan Mathison Turing was born in Maida Vale, [London] on 23 June 1912. His father was part of a family of merchants from Scotland. His mother, Ethel Sara, was the daughter of an engineer.\n\nEducation \nTuring went to St. Michael\'s, a school at 20 Charles Road, St Leonards-on-sea, when he was five years old.\n"This is only a foretaste of what is to come, and only the shadow of what is going to be.” – Alan Turing.\n\nThe Stoney family were once prominent landlords in North Tipperary. His mother Ethel Sara Stoney (1881–1976) was daughter of Edward Waller Stoney (Borrisokane, North Tipperary) and Sarah Crawford (Cartron Abbey, Co. Longford), who were Protestant Anglo-Irish gentry. She was educated in Dublin at Alexandra School and College. On October 1st 1907 she married Julius Mathison T

In [11]:
# check the token count of each chunk, to confirm they're within our chunk_size limit
toke_counts = [tiktoken_len(chunk) for chunk in chunks]

toke_counts

[305, 383, 381, 82]

Using the `text_splitter` we get much better sized chunks of text. We'll use this functionality during the indexing process later. Now let's take a look at embedding.



<br>

### Generate embeddings

Now that we can split our articles into well-sized chunks, we need a way to make those chunks searchable by *meaning* rather than just by keyword.

We do this by converting each chunk of text into an **embedding**: a numerical vector that captures the semantic meaning of the text. Chunks with similar meaning end up with similar embeddings, which is what will later let us find the most relevant chunks for a given question by comparing vectors instead of raw text.

For this demo, we'll use the `text-embedding-ada-002` embedding model from OpenAI (available through their API):

In [12]:
from langchain_openai import OpenAIEmbeddings

model_name = 'text-embedding-ada-002'

embed = OpenAIEmbeddings(
    model=model_name,
    openai_api_key=OPENAI_API_KEY
)

Now we embed some text like so:

In [ ]:
texts = [
    'this is the first chunk of text',
    'then another second chunk of text is here'
]

# embed_documents() sends a list of texts to the embedding model and returns
# one embedding vector per input text
res = embed.embed_documents(texts)

print(f"Number of embeddings generated: {len(res)}")
print(f"Dimensionality of each embedding vector: {len(res[0])}") # (1536 for this model)

print("\n\nThis is how the embedding for the first document looks like... ")
print(res[0])

Number of embeddings generated: 2
Dimensionality of each embedding vector: 1536



<br>

### Store in a vector database

To create our vector database we first need an API key from Pinecone.

> 💡 **Note:** Get a free API key at [app.pinecone.io](https://app.pinecone.io), then add it to your `.env` file as:
> ```
> PINECONE_API_KEY=your-key-here
> ```

In [14]:
import os

from pinecone import Pinecone, ServerlessSpec

pinecone_api_key = os.environ.get("PINECONE_API_KEY")
if not pinecone_api_key:
    raise ValueError(
        "PINECONE_API_KEY is not set. Add it to your .env file."
    )

# configure client
pc = Pinecone(api_key=pinecone_api_key)

Now we setup our index specification, this allows us to define the cloud provider and region where we want to deploy our index. You can find a list of all [available providers and regions here](https://docs.pinecone.io/docs/projects).

In [15]:
from pinecone import ServerlessSpec

spec = ServerlessSpec(
    cloud="aws", region="us-east-1"
)

Then we initialize the index. We will be using OpenAI's `text-embedding-ada-002` model for creating the embeddings, so we set the `dimension` to `1536`.

In [16]:
import time

index_name = 'langchain-retrieval-augmentation'  # change if desired

existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=1536,  # Number of values in each embedding vector (must match the embedding model)
        metric="cosine", # Use cosine similarity to compare vectors by their direction (higher similarity = more relevant match)
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

index = pc.Index(index_name)

<br>

### Indexing the full dataset

Now we put everything together and index the entire dataset (all 10,000 articles) into Pinecone.

For each article we will:

1. **Split** the article's text into chunks, using the `text_splitter` we defined earlier.
2. **Attach metadata** to each chunk (its source article's `wiki-id`, `source` URL, `title`, the chunk's text, and its position within the article), so we can trace it back later.
3. **Embed** the chunks in batches (of `100` or more) using our `embed` model.
4. **Upsert** the embeddings, along with their metadata, into our Pinecone index. 
    - An *Upsert* (update + insert) is a database operation that inserts a new row if it doesn’t exist or updates it if it does, based on a unique key or condition. It combines "update" and "insert" into a single action.
    - In this case, we'll insert each record if its ID doesn't exist yet, or updated if it does — so we can safely re-run this process without creating duplicates.

We process articles in batches, rather than one at a time, to reduce the number of API calls to the embedding model and to Pinecone, which makes indexing significantly faster.



In [17]:
#
# note: running this cell will take some time because it splits the text,
# generates embeddings for each chunk, and uploads them to the vector index.
#

from tqdm.auto import tqdm
from uuid import uuid4

batch_limit = 100

texts = []
metadatas = []

for i, record in enumerate(tqdm(data)):
    # first get metadata fields for this record
    metadata = {
        'wiki-id': str(record['id']),
        'source': record['url'],
        'title': record['title']
    }
    # now we create chunks from the record text
    record_texts = text_splitter.split_text(record['text'])
    # create individual metadata dicts for each chunk
    record_metadatas = [{
        "chunk": j, "text": text, **metadata
    } for j, text in enumerate(record_texts)]
    # append these to current batches
    texts.extend(record_texts)
    metadatas.extend(record_metadatas)
    # if we have reached the batch_limit we can add texts
    if len(texts) >= batch_limit:
        ids = [str(uuid4()) for _ in range(len(texts))]
        embeds = embed.embed_documents(texts)
        index.upsert(vectors=zip(ids, embeds, metadatas))
        texts = []
        metadatas = []

if len(texts) > 0:
    ids = [str(uuid4()) for _ in range(len(texts))]
    embeds = embed.embed_documents(texts)
    index.upsert(vectors=zip(ids, embeds, metadatas))

100%|██████████| 10000/10000 [06:56<00:00, 23.98it/s]


We've now indexed everything. We can check the number of vectors in our index like so:

In [18]:
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 58098}},
 'total_vector_count': 58098}

The output above tells us a few things about our index:

- `dimension: 1536` — each vector stored in the index has 1536 values, matching the `text-embedding-ada-002` embedding model we used.
- `index_fullness: 0.0` — a rough estimate of how full the index is relative to its capacity; `0.0` means we're nowhere near any storage limits.
- `namespaces: {'': {'vector_count': 29049}}` — our vectors live in the default (unnamed) namespace, which contains `29,049` vectors (i.e. chunks) in total.
- `total_vector_count: 29049` — the total number of vectors across all namespaces, confirming all `29,049` chunks were successfully indexed.

<br>

### Creating a Vector Store and Querying

Now that our index is populated, we can wrap it in a LangChain `PineconeVectorStore`, pointing it at our existing Pinecone `index` and the `embed` model used to create the embeddings.

Once we have a `vector_store`, we can query it directly with `similarity_search()`: LangChain embeds our query text under the hood, uses it to search the index, and returns the `k` most relevant chunks (here, `k=3`) — no need to manually embed the query or call the Pinecone client ourselves.

In [19]:
from langchain_pinecone import PineconeVectorStore

vector_store = PineconeVectorStore(index=index, embedding=embed, text_key="text")


In [20]:
query = "who was Alan Turing?"

vector_store.similarity_search(
    query,  # our search query
    k=3  # return 3 most relevant docs
)

[Document(metadata={'chunk': 1.0, 'source': 'https://simple.wikipedia.org/wiki/Alan%20Turing', 'title': 'Alan Turing', 'wiki-id': '13'}, page_content='Alan was a brilliant mathematician and cryptographer. He became the founder of modern-day computer science and artificial intelligence. He designed a machine at Bletchley Park to break secret Enigma encrypted messages used by the Nazi German war machine to protect sensitive commercial, diplomatic and military communications during World War 2. This made the single biggest contribution to the Allied victory in the war against Nazi Germany. It possibly saved the lives of an estimated 2 million people, and shortened World War II.\n\nIn 2013, almost 60 years later, Turing received a posthumous Royal Pardon from Queen Elizabeth II. Today, the “Turing law” grants an automatic pardon to men who died before the law came into force, making it possible for living convicted gay men to seek pardons for offences now no longer on the statute book.\n\n

All of these are good, relevant results. But what can we do with this? There are many tasks, one of the most interesting (and well supported by LangChain) is called _"Generative Question-Answering"_ or GQA.



<br>

## Generative Question-Answering (GQA)

So far, `similarity_search()` only *retrieves* the raw chunks of text most relevant to a query — it's up to us to read through them and work out the answer ourselves.

**Generative Question-Answering (GQA)** takes this a step further by automating that last step: instead of just returning matching chunks, we hand them to an **LLM** along with the original question, and let the model *generate* a natural-language answer grounded in that retrieved context. This is the "Generation" part of Retrieval-Augmented Generation.

LangChain provides ready-made chains for this exact pattern, so we don't have to manually stitch together the retrieved chunks and the question into a prompt ourselves. Below we'll use `RetrievalQA`, one of LangChain's built-in chains, which:

1. Takes a user question.
2. Uses our `vector_store` as a **retriever** to fetch the most relevant chunks.
3. **Stuffs** those chunks into a prompt template together with the question (hence `chain_type="stuff"`).
4. Sends that prompt to the LLM and returns the generated answer.


In [21]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

# completion llm
llm = ChatOpenAI(
    openai_api_key=OPENAI_API_KEY,
    model_name='gpt-3.5-turbo',
    temperature=0.0
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever()
)

In [22]:
qa.invoke(query)

{'query': 'who was Alan Turing?',
 'result': "Alan Turing was an English mathematician, computer scientist, and cryptographer. He is considered the founder of modern-day computer science and artificial intelligence. Turing played a crucial role during World War II by designing a machine to break secret Enigma encrypted messages used by the Nazi German war machine, which significantly contributed to the Allied victory. He also created the theoretical Turing machine and proposed the Turing test to determine machine intelligence. Turing's life was tragically cut short in 1954, and he is known for his significant contributions to mathematics, computer science, and codebreaking."}

We can also include the sources of information that the LLM is using to answer our question. We can do this using a slightly different version of `RetrievalQA` called `RetrievalQAWithSourcesChain`:

In [23]:
from langchain.chains import RetrievalQAWithSourcesChain

qa_with_sources = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_store.as_retriever()
)

In [24]:
qa_with_sources.invoke(query)

{'question': 'who was Alan Turing?',
 'answer': "Alan Turing was an English mathematician and computer scientist who played a crucial role in breaking the Enigma code during World War II and is considered the founder of modern-day computer science and artificial intelligence. He was born in Maida Vale, London in 1912 and tragically died in 1954. Turing's work significantly contributed to the Allied victory in World War II and his legacy continues to impact the field of computer science today.\n",
 'sources': 'https://simple.wikipedia.org/wiki/Alan%20Turing'}

Now we answer the question being asked, *and* return the source of this information being used by the LLM.

Delete the index to save resources when you're done!

In [25]:
# RECOMMENDED: delete the index to save resources once you're done with this demo
#
#  Pinecone indexes consume storage/compute resources even when idle, so it's good
#  practice to delete them when you no longer need them (especially on free-tier plans).
#
# - Uncomment the line below to delete the index.
# - If you're planning to keep experimenting or sending more queries later, leave it
#   commented out so you don't have to rebuild the index (re-embed and re-upsert
#   all 10,000 articles) from scratch.
#

# pc.delete_index(index_name)

---

Now we have answered the question being asked but also included the source of this information being used by the LLM.

We’ve learned how to ground Large Language Models with source knowledge by using a vector database as our knowledge base. Using this, we can encourage accuracy in our LLM’s responses, keep source knowledge up to date, and improve trust in our system by providing citations with every answer.

We’re already seeing LLMs and knowledge bases paired together in huge products like Bing’s AI search, Google Bard, and ChatGPT plugins. Without a doubt, the future of LLMs is tightly coupled with high-performance, scalable, and reliable knowledge bases.

<br>

## Summary - Indexing in LangChain

Indexing in LangChain refers to the process of organizing and preparing large datasets or corpora so that they can be efficiently searched and retrieved. LangChain, a framework for developing applications that involve complex language processing, utilizes indexing to manage and optimize access to textual data, especially when dealing with extensive documents or conversational histories.

<br>

### Key Concepts of Indexing in LangChain

1. **Purpose of Indexing**:
   - **Efficiency**: To enable quick search and retrieval of relevant information from large datasets.
   - **Scalability**: To handle large volumes of data without compromising performance.
   - **Optimization**: To facilitate the use of advanced language models and algorithms by structuring data appropriately.

2. **Types of Indexing**:
   - **Textual Indexing**: Creating indices for text documents to allow keyword-based or semantic searches.
   - **Vector Indexing**: Utilizing vector embeddings (numerical representations of text) to enable similarity searches, which are crucial for tasks like information retrieval and question-answering.

3. **Indexing Methods in LangChain**:
   - **Keyword Indexing**: Creating inverted indices that map keywords to their locations in the dataset, similar to traditional search engines.
   - **Embedding-Based Indexing**: Using embeddings generated by language models (e.g., BERT, GPT) to represent text in a high-dimensional space. This allows for similarity searches using techniques like nearest neighbor search.

4. **Implementation in LangChain**:
   - LangChain provides tools and utilities to create and manage indices. These tools can be integrated with various backend systems (e.g., Elasticsearch, FAISS) to leverage their indexing capabilities.
   - **Custom Indexers**: Users can implement custom indexing strategies tailored to their specific needs and integrate them into their LangChain pipelines.

